# Audio DiT — TPU v5e-8 Training (PyTorch/XLA)

Launcher notebook for Kaggle. Select **TPU v5e-8** as the accelerator.

**Before running, attach two Kaggle Datasets to this notebook:**
1. `audio-dit-code` — the `audio_dit/` source files (`config.py`, `dit.py`, `diffusion.py`, `dataset.py`, `train.py`, `train_xla.py`)
2. `audiocaps-precomputed` — the cache produced by `precompute.py` (the `train/` folder with `shard_*.pt` + `meta.json`)

Training only needs `torch` + `torch_xla`, both preinstalled on Kaggle TPU images — **no pip installs required** (installing `torch`/`torchaudio` here can break the XLA build; the heavy HF libraries are only used at precompute/inference time, not here).

Kaggle sessions are time-limited and preemptible. Checkpoints save locally every `ckpt_every` steps, prune older files, and upload to Hugging Face every `UPLOAD_EVERY` saves. Losses are logged to Weights & Biases.

**Kaggle Secrets required** (Add-ons → Secrets): `HF_TOKEN` (write access), `WANDB_API_KEY`.

In [ ]:
# Do NOT reinstall torch / torch_xla on Kaggle TPU — the image already has a
# matching pair and pip can break XLA. Only install the extras we need.
!pip install -q huggingface_hub wandb


In [1]:
# Sanity check: confirm the TPU is visible before spending session time.
import torch
import torch_xla
import torch_xla.core.xla_model as xm

print("torch     :", torch.__version__)
print("torch_xla :", torch_xla.__version__)
print("devices   :", xm.get_xla_supported_devices())  # expect 8 TPU cores on v5e-8

ImportError: /usr/local/lib/python3.12/dist-packages/_XLAC.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN5torch8autograd13_wrap_outputsERKSt6vectorIN2at6TensorESaIS3_EERKSt13unordered_setIPN3c1010TensorImplESt4hashISB_ESt8equal_toISB_ESaISB_EESJ_NS9_8ArrayRefISt8optionalIS3_EEERKSt10shared_ptrINS0_4NodeEERKSt8functionIFS5_S5_S5_EESJ_RKST_IFS3_S3_EE

## Stage the code and locate the precomputed cache

`train_xla.py` forks 8 processes (one per chip), so it must run as a **script via `!python`**, not as notebook cells — `torch_xla.launch` cannot fork from an interactive kernel.

In [ ]:
# Paths + training I/O config. Adjust DATA_DIR to your attached Kaggle Dataset.
import os

DATA_DIR = "/kaggle/input/datasets/sidharthangn/audiocaps-precomputed-cache"
OUT_DIR = "/kaggle/working/runs/dit_b2"
RESUME = None          # e.g. "/kaggle/working/runs/dit_b2/ckpt_0002000.pt"
WORKERS = 4

# ---- Hugging Face checkpoint backup ----
# Create a write-token secret named HF_TOKEN in Kaggle → Add-ons → Secrets.
HF_REPO = "Sidharthan/QADiT-checkpoints"
HF_PATH_IN_REPO = "runs/dit_b2"
UPLOAD_EVERY = 5       # upload every N local checkpoint saves
KEEP_LOCAL = 1         # keep only the newest ckpt_*.pt on /kaggle/working

# ---- Weights & Biases ----
# Create a secret named WANDB_API_KEY in Kaggle → Add-ons → Secrets.
WANDB_PROJECT = "audio-dit"
WANDB_RUN_NAME = "dit_b2_tpu_v5e8"
WANDB_ENABLED = True

# Load secrets (Kaggle). Falls back to env vars for local runs.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    HF_TOKEN = _secrets.get_secret("HUGGINGFACE_TOKEN")
    os.environ["WANDB_API_KEY"] = _secrets.get_secret("WandB_TOKEN")
except Exception as e:
    print(f"[warn] kaggle_secrets unavailable ({e}); using env vars / empty token")
    HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN") or ""

assert os.path.exists(f"{DATA_DIR}/train/meta.json"), \
    "precomputed cache not found - attach the dataset or fix DATA_DIR"
print("cache OK:", DATA_DIR)
print("HF_REPO :", HF_REPO, "| token set:", bool(HF_TOKEN))
print("W&B     :", WANDB_PROJECT, "/", WANDB_RUN_NAME, "| enabled:", WANDB_ENABLED)


cache OK: /kaggle/input/datasets/sidharthangn/audiocaps-precomputed-cache


## Import required modules

In [ ]:
import math
import argparse
import time
import json
import copy
import shutil
from pathlib import Path
import os

from huggingface_hub import HfApi, create_repo
import wandb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.runtime as xr


## Huggingface Checkpoint Uploader

In [ ]:
def upload_ckpt_folder(
    local_dir: str | Path,
    repo_id: str,
    token: str,
    path_in_repo: str | None = None,
    commit_message: str = "upload checkpoint",
):
    """
    Upload a local folder (or a single checkpoint directory) to a HF model repo.
    repo_id example: "your-username/audio-dit-checkpoints"
    path_in_repo: subfolder inside the repo, e.g. "runs/dit_b2"
    """
    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise FileNotFoundError(local_dir)
    api = HfApi(token=token)
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=token)
    api.upload_folder(
        folder_path=str(local_dir),
        repo_id=repo_id,
        repo_type="model",
        path_in_repo=path_in_repo,
        commit_message=commit_message,
        # Ignore temp / incomplete files if any
        ignore_patterns=["*.tmp", "*.lock"],
    )
    print(f"[hf] uploaded {local_dir} -> {repo_id}"
          + (f"/{path_in_repo}" if path_in_repo else ""))

## Weights & Biases helpers

Do **not** call `wandb.init()` in the notebook process. XLA forks 8 workers —
only the **master ordinal** should create the W&B run (inside `_mp_fn`).


In [ ]:
def init_wandb(cfg, is_master: bool, project: str = "audio-dit",
               run_name: str | None = None, enabled: bool = True,
               resume_id: str | None = None):
    """Create the W&B run on the master ordinal only."""
    if not (enabled and is_master):
        return None
    run = wandb.init(
        project=project,
        name=run_name or "dit_b2_tpu_v5e8",
        config=cfg.as_dict(),
        resume="allow" if resume_id else None,
        id=resume_id,
    )
    print(f"[wandb] run: {run.url}", flush=True)
    return run


def make_log_fn(is_master, start_step, t0, sched, use_wandb: bool = True):
    """XLA-safe logger used via xm.add_step_closure (print + wandb)."""
    def log_fn(step, loss, diff, repa, lam):
        # Materialise tensors HERE — the graph has already executed.
        loss_v = float(loss)
        diff_v = float(diff)
        repa_v = float(repa)
        lam_v = float(lam)
        lr = sched.get_last_lr()[0]
        rate = (step - start_step + 1) / max(time.time() - t0, 1e-6)

        if not is_master:
            return

        print(
            f"step {step:>7d} | loss {loss_v:.4f} "
            f"(diff {diff_v:.4f}, repa {repa_v:.4f}, lam {lam_v:.3f}) "
            f"| lr {lr:.2e} | {rate:.2f} it/s/chip",
            flush=True,
        )
        if use_wandb and wandb.run is not None:
            wandb.log(
                {
                    "train/loss_total": loss_v,
                    "train/loss_diff": diff_v,
                    "train/loss_repa": repa_v,
                    "train/repa_lambda": lam_v,
                    "train/lr": lr,
                    "perf/it_per_s_per_chip": rate,
                },
                step=step,
            )
    return log_fn


print("[wandb] helpers ready")


## Config

In [8]:
"""
Central configuration for the text-to-audio latent DiT.

Everything that must stay CONSISTENT between precompute.py, train.py and
sample.py lives here.  The three stages share one source of truth so the
"locked triple" (mel config <-> VAE <-> vocoder) can never silently drift.

Shape conventions used throughout the project
---------------------------------------------
waveform : [B, 163840]           10.24 s of 16 kHz mono audio
mel      : [B, 1, 1024, 64]      (channel, TIME frames, MEL bins) - this is the
                                 orientation the AudioLDM VAE expects (time is
                                 the "image height", mel bins the "width")
latent   : [B, 8, 256, 16]       VAE downsamples time and freq by 4x
DiT grid : (128, 8) = 1024 tokens after 2x2 patchify (time-major flatten:
                                 token n = t * 8 + f)
text     : [B, 64, 1024]         FLAN-T5-large hidden states + [B, 64] mask
REPA y*  : [B, 1024, 768]        AST features bilinearly resampled onto the
                                 exact same 1024-token DiT grid at precompute
"""

from dataclasses import dataclass, field, asdict


# --------------------------------------------------------------------------- #
#  Audio front-end  (LOCKED to the AudioLDM VAE + HiFi-GAN vocoder pair)      #
# --------------------------------------------------------------------------- #
@dataclass
class MelConfig:
    sample_rate: int = 16_000
    duration_s: float = 10.24          # 10.24 s -> exactly 1024 mel frames
    n_fft: int = 1024
    win_length: int = 1024
    hop_length: int = 160              # 10 ms hop at 16 kHz
    n_mels: int = 64
    f_min: float = 0.0
    f_max: float = 8_000.0

    @property
    def num_samples(self) -> int:      # 163_840 samples per clip
        return int(self.sample_rate * self.duration_s)

    @property
    def num_frames(self) -> int:       # 1024 mel frames per clip
        return self.num_samples // self.hop_length


# --------------------------------------------------------------------------- #
#  Frozen pretrained components (HuggingFace ids)                              #
# --------------------------------------------------------------------------- #
@dataclass
class PretrainedConfig:
    # AudioLDM ships a matched (VAE, vocoder) pair for the mel config above.
    vae_repo: str = "cvssp/audioldm-s-full-v2"     # subfolder="vae"
    vocoder_repo: str = "cvssp/audioldm-s-full-v2" # subfolder="vocoder"
    # Text encoder: token-level embeddings for cross-attention (Tango recipe).
    text_model: str = "google/flan-t5-large"       # hidden size 1024
    text_max_len: int = 64
    text_dim: int = 1024
    # REPA target: AST = Audio Spectrogram Transformer (audio analog of the
    # image SSL encoders REPA was designed around).  Hidden size 768.
    repa_model: str = "MIT/ast-finetuned-audioset-10-10-0.4593"
    repa_dim: int = 768


# --------------------------------------------------------------------------- #
#  Latent geometry (determined by the VAE, must match precompute output)      #
# --------------------------------------------------------------------------- #
@dataclass
class LatentConfig:
    channels: int = 8      # AudioLDM KL-VAE latent channels
    time: int = 256        # 1024 mel frames / 4x VAE downsample
    freq: int = 16         # 64 mel bins    / 4x VAE downsample


# --------------------------------------------------------------------------- #
#  DiT backbone                                                                #
# --------------------------------------------------------------------------- #
@dataclass
class DiTConfig:
    patch_size: int = 2      # 2x2 patches over the (256, 16) latent -> 1024 tokens
    hidden_size: int = 768   # DiT-B width
    depth: int = 12          # DiT-B depth
    num_heads: int = 12
    mlp_ratio: float = 4.0
    # REPA taps the hidden state after this block (~depth/3, per the paper).
    repa_layer: int = 4


# --------------------------------------------------------------------------- #
#  Diffusion process                                                           #
# --------------------------------------------------------------------------- #
@dataclass
class DiffusionConfig:
    num_train_steps: int = 1000     # discrete timesteps T
    schedule: str = "cosine"        # cosine alpha-bar schedule
    # Training-time timestep sampling: logit-normal concentrates samples in the
    # informative mid-noise regime (SD3 trick) instead of uniform.
    logit_normal_mean: float = 0.0
    logit_normal_std: float = 1.0
    # Classifier-free guidance
    p_uncond: float = 0.1           # caption dropout probability at train time
    cfg_scale: float = 4.0          # default guidance weight at sampling time
    sample_steps: int = 50          # DDIM steps at inference


# --------------------------------------------------------------------------- #
#  Training                                                                    #
# --------------------------------------------------------------------------- #
@dataclass
class TrainConfig:
    batch_size: int = 32
    lr: float = 1e-4
    weight_decay: float = 0.0
    warmup_steps: int = 1000
    total_steps: int = 200_000
    ema_decay: float = 0.9999
    grad_clip: float = 1.0
    # REPA loss weight: starts at repa_weight, linearly decays to 0 over
    # repa_decay_steps (REPA helps most early; let the diffusion loss own the
    # end of training).  Set repa_weight=0.0 to disable REPA entirely (A/B).
    repa_weight: float = 0.5
    repa_decay_steps: int = 200_000
    log_every: int = 50
    ckpt_every: int = 2000
    seed: int = 0


# --------------------------------------------------------------------------- #
#  Top-level bundle                                                            #
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    mel: MelConfig = field(default_factory=MelConfig)
    pretrained: PretrainedConfig = field(default_factory=PretrainedConfig)
    latent: LatentConfig = field(default_factory=LatentConfig)
    dit: DiTConfig = field(default_factory=DiTConfig)
    diffusion: DiffusionConfig = field(default_factory=DiffusionConfig)
    train: TrainConfig = field(default_factory=TrainConfig)

    def as_dict(self) -> dict:
        return asdict(self)

    # ------------------------------------------------------------------ #
    # A tiny configuration whose forward pass runs in seconds on CPU.     #
    # Used by `train.py --smoke` to verify the whole pipeline end-to-end  #
    # (shapes, losses, sampler) without any real data or pretrained nets. #
    # ------------------------------------------------------------------ #
    @classmethod
    def smoke(cls) -> "Config":
        cfg = cls()
        cfg.latent = LatentConfig(channels=4, time=32, freq=8)
        cfg.dit = DiTConfig(patch_size=2, hidden_size=64, depth=3,
                            num_heads=4, mlp_ratio=2.0, repa_layer=1)
        cfg.pretrained.text_dim = 32
        cfg.pretrained.text_max_len = 8
        cfg.pretrained.repa_dim = 16
        cfg.diffusion.sample_steps = 5
        cfg.train.batch_size = 4
        cfg.train.warmup_steps = 2
        cfg.train.total_steps = 20
        cfg.train.repa_decay_steps = 20
        cfg.train.log_every = 1
        cfg.train.ckpt_every = 10
        return cfg


## Dataset Preprocessor

In [9]:
"""
Dataset over the shards written by precompute.py.

Each shard is a single .pt file holding a few hundred samples as stacked
tensors, so reads are large and sequential (friendly to spinning disks, cloud
buckets and, later, to a grain/tf.data port on TPU).  Shards are cached in
memory with a tiny LRU so random access across shard boundaries stays cheap.
"""

class PrecomputedAudioCaps(Dataset):
    """Yields dicts with keys: latent, text_emb, text_mask, repa, caption.

    `latent` is returned ALREADY SCALED to ~unit variance (multiplied by
    meta.json's latent_scale), so train.py can treat it as diffusion-ready.
    """

    def __init__(self, root: str, split: str = "train", cache_shards: int = 2):
        self.dir = Path(root) / split
        meta_path = self.dir / "meta.json"
        if not meta_path.exists():
            raise FileNotFoundError(
                f"{meta_path} not found - run precompute.py --split {split} first")
        with open(meta_path) as f:
            self.meta = json.load(f)

        self.latent_scale = float(self.meta["latent_scale"])
        self.shard_files = [self.dir / name for name in self.meta["shards"]]

        # Index: global sample idx -> (shard idx, offset inside shard).
        # Shard sizes can vary (the last one is usually short), so read the
        # true length of each shard once up front.
        self.index: list[tuple[int, int]] = []
        for si, path in enumerate(self.shard_files):
            n = torch.load(path, map_location="cpu")["latent"].shape[0]
            self.index.extend((si, oi) for oi in range(n))

        self.cache_shards = cache_shards
        self._cache: dict[int, dict] = {}    # shard idx -> loaded dict (LRU)

    def _shard(self, si: int) -> dict:
        if si not in self._cache:
            if len(self._cache) >= self.cache_shards:
                self._cache.pop(next(iter(self._cache)))   # evict oldest
            self._cache[si] = torch.load(self.shard_files[si], map_location="cpu")
        return self._cache[si]

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, idx: int) -> dict:
        si, oi = self.index[idx]
        shard = self._shard(si)
        return {
            # fp16 on disk -> fp32 for the training math; scaled to unit var.
            "latent": shard["latent"][oi].float() * self.latent_scale,
            "text_emb": shard["text_emb"][oi].float(),
            "text_mask": shard["text_mask"][oi].long(),
            "repa": shard["repa"][oi].float(),
            "caption": shard["caption"][oi],
        }


def collate(batch: list[dict]) -> dict:
    """Stack tensors; keep captions as a plain list of strings."""
    return {
        "latent": torch.stack([b["latent"] for b in batch]),
        "text_emb": torch.stack([b["text_emb"] for b in batch]),
        "text_mask": torch.stack([b["text_mask"] for b in batch]),
        "repa": torch.stack([b["repa"] for b in batch]),
        "caption": [b["caption"] for b in batch],
    }


In [13]:
cfg = Config()

In [15]:
ds = PrecomputedAudioCaps(DATA_DIR, "train")
loader = DataLoader(ds, batch_size=cfg.train.batch_size, shuffle=True,
                            num_workers=2,
                            collate_fn=collate, drop_last=True)

def infinite():
        while True:
                yield from loader

it = infinite()
get_batch = lambda: next(it)

get_batch()


{'latent': tensor([[[[ 7.1346e-02,  1.2529e+00,  1.6510e+00,  ...,  5.8562e-01,
             5.9026e-01, -3.9002e-01],
           [-1.2715e-01,  1.1508e+00,  1.3058e+00,  ...,  5.8562e-01,
             6.9188e-01, -1.8573e-01],
           [ 1.4747e+00,  1.1749e+00,  1.1350e+00,  ...,  1.6232e+00,
             1.0487e+00,  1.7819e-01],
           ...,
           [-6.3146e+00, -3.8200e+00, -4.0390e+00,  ..., -4.0910e+00,
            -4.0910e+00, -4.8037e+00],
           [-3.8274e+00, -1.6269e+00, -1.8209e+00,  ..., -1.9267e+00,
            -2.8752e+00, -4.7035e+00],
           [-3.6585e+00, -3.3541e+00, -3.3448e+00,  ..., -2.9828e+00,
            -3.5174e+00, -4.8037e+00]],
 
          [[-1.0422e+00,  1.8295e-01,  5.0377e-02,  ..., -5.9072e-01,
            -1.8643e-01, -1.7606e+00],
           [ 8.0928e-01, -5.3921e-01, -1.0135e+00,  ..., -3.3643e-01,
             2.7842e-01, -8.1624e-01],
           [-8.6079e-01, -9.4896e-01,  4.9838e-01,  ...,  3.0441e-01,
             1.2715e+00, -8.0

In [17]:
get_batch()["latent"].shape

torch.Size([32, 8, 256, 16])

## Diffusion (DDIM + v Prediction)

In [ ]:
"""
Diffusion process - schedules, forward noising, v-prediction, DDIM sampling.

Everything here is plain tensor math (no diffusers).  We use:

  * a COSINE alpha-bar schedule (Nichol & Dhariwal, 2021),
  * V-PREDICTION as the network target (Salimans & Ho, 2022):
        v = sqrt(abar_t) * eps - sqrt(1 - abar_t) * z0
    which trains more stably than raw epsilon for transformer backbones and
    interpolates between predicting noise (t small) and data (t large),
  * LOGIT-NORMAL timestep sampling at train time (SD3): more probability mass
    on mid-range noise levels where the model actually learns,
  * a DDIM sampler with CLASSIFIER-FREE GUIDANCE for inference.

Key identities used throughout (see DDPM.md notes):
    z_t  = sqrt(abar_t) * z0 + sqrt(1 - abar_t) * eps      (forward jump)
    z0   = sqrt(abar_t) * z_t - sqrt(1 - abar_t) * v       (recover data)
    eps  = sqrt(1 - abar_t) * z_t + sqrt(abar_t) * v       (recover noise)
"""

import math

import torch


class Diffusion:
    """Holds the schedule tables and implements q(z_t|z0), targets, sampling."""

    def __init__(self, num_train_steps: int = 1000, schedule: str = "cosine",
                 logit_normal_mean: float = 0.0, logit_normal_std: float = 1.0):
        self.T = num_train_steps
        self.ln_mean = logit_normal_mean
        self.ln_std = logit_normal_std

        if schedule == "cosine":
            # abar(t) = cos^2( (t/T + s) / (1 + s) * pi/2 ), s = 0.008.
            # Clipped so abar never reaches exactly 0 or 1 (numerical safety).
            s = 0.008
            steps = torch.arange(self.T + 1, dtype=torch.float64)
            f = torch.cos((steps / self.T + s) / (1 + s) * math.pi / 2) ** 2
            abar = (f / f[0]).clamp(1e-5, 1.0)
            self.alpha_bar = abar[1:].float()          # abar at t = 1..T -> index t-1
        else:
            raise ValueError(f"unknown schedule: {schedule}")

    def to(self, device) -> "Diffusion":
        """Move the schedule table to the accelerator ONCE.

        On XLA, leaving alpha_bar on CPU would re-upload it (host->device
        transfer) inside every training step; pinning it on the device keeps
        the step graph free of host traffic.
        """
        self.alpha_bar = self.alpha_bar.to(device)
        return self

    # ------------------------------------------------------------------ #
    #  Schedule lookups                                                    #
    # ------------------------------------------------------------------ #
    def _gather(self, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """abar coefficients for integer timesteps t in [0, T), broadcastable
        against a [B, C, T, F] latent.  Returns (sqrt_abar, sqrt_one_minus)."""
        abar = self.alpha_bar.to(t.device)[t]          # [B]
        sqrt_abar = abar.sqrt().view(-1, 1, 1, 1)
        sqrt_1m = (1 - abar).sqrt().view(-1, 1, 1, 1)
        return sqrt_abar, sqrt_1m

    # ------------------------------------------------------------------ #
    #  Training utilities                                                  #
    # ------------------------------------------------------------------ #
    def sample_timesteps(self, batch_size: int, device) -> torch.Tensor:
        """Logit-normal timestep sampling.

        u ~ N(mean, std); sigmoid(u) in (0,1) is mapped to an integer step.
        Compared to uniform sampling this concentrates training on mid-noise
        timesteps, which measurably speeds up DiT convergence.
        """
        u = torch.randn(batch_size, device=device) * self.ln_std + self.ln_mean
        frac = torch.sigmoid(u)
        return (frac * self.T).long().clamp(0, self.T - 1)

    def add_noise(self, z0: torch.Tensor, t: torch.Tensor,
                  eps: torch.Tensor) -> torch.Tensor:
        """Forward process q(z_t | z0): jump straight to noise level t."""
        sqrt_abar, sqrt_1m = self._gather(t)
        return sqrt_abar * z0 + sqrt_1m * eps

    def v_target(self, z0: torch.Tensor, t: torch.Tensor,
                 eps: torch.Tensor) -> torch.Tensor:
        """The v-prediction regression target."""
        sqrt_abar, sqrt_1m = self._gather(t)
        return sqrt_abar * eps - sqrt_1m * z0

    # ------------------------------------------------------------------ #
    #  Conversions (used by the sampler)                                   #
    # ------------------------------------------------------------------ #
    def z0_from_v(self, z_t: torch.Tensor, t: torch.Tensor,
                  v: torch.Tensor) -> torch.Tensor:
        sqrt_abar, sqrt_1m = self._gather(t)
        return sqrt_abar * z_t - sqrt_1m * v

    def eps_from_v(self, z_t: torch.Tensor, t: torch.Tensor,
                   v: torch.Tensor) -> torch.Tensor:
        sqrt_abar, sqrt_1m = self._gather(t)
        return sqrt_1m * z_t + sqrt_abar * v

    # ------------------------------------------------------------------ #
    #  DDIM sampling with classifier-free guidance                         #
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def ddim_sample(self, model, shape: tuple, text_emb: torch.Tensor,
                    text_mask: torch.Tensor, num_steps: int = 50,
                    cfg_scale: float = 4.0, eta: float = 0.0,
                    device: str = "cpu",
                    generator: torch.Generator | None = None) -> torch.Tensor:
        """Generate latents from pure noise, guided by text.

        model     : the (EMA) DiT - called twice per step (cond + uncond) when
                    cfg_scale > 1, batched together for efficiency.
        shape     : [B, C, T, F] latent shape to generate.
        eta       : 0.0 = deterministic DDIM; 1.0 = ancestral (DDPM-like).
        Returns z0-hat in the SCALED latent space (divide by the latent scale
        factor before VAE decoding - sample.py handles that).
        """
        B = shape[0]
        z = torch.randn(shape, device=device, generator=generator)

        # Evenly spaced timestep subsequence T-1 ... 0 (e.g. 50 of 1000 steps).
        times = torch.linspace(self.T - 1, 0, num_steps, device=device).long()

        # CFG needs a conditional and an unconditional prediction per step.
        # The unconditional branch uses the model's learned null context
        # (text_emb=None path).  We keep the two passes separate for clarity;
        # batching them into one 2B forward is a straightforward optimisation.
        use_cfg = cfg_scale is not None and cfg_scale > 1.0

        for i in range(num_steps):
            t = times[i].expand(B)                       # current step, [B]

            if use_cfg:
                # Two predictions: with caption and with the learned null.
                v_cond = model(z, t.float(), text_emb, text_mask)
                v_uncond = model(z, t.float(), None, None)
                # CFG extrapolation in v-space (linear in the prediction):
                v = v_uncond + cfg_scale * (v_cond - v_uncond)
            else:
                v = model(z, t.float(), text_emb, text_mask)

            # Convert the v prediction into (z0-hat, eps-hat) at this level.
            z0_hat = self.z0_from_v(z, t, v)
            eps_hat = self.eps_from_v(z, t, v)

            if i == num_steps - 1:
                z = z0_hat                               # final step: output data
                break

            # DDIM update to the next (lower) noise level.
            t_next = times[i + 1].expand(B)
            abar_next = self.alpha_bar.to(device)[t_next].view(-1, 1, 1, 1)
            abar_now = self.alpha_bar.to(device)[t].view(-1, 1, 1, 1)

            # Optional stochasticity (eta > 0 recovers DDPM-like sampling).
            sigma = eta * torch.sqrt((1 - abar_next) / (1 - abar_now)
                                     * (1 - abar_now / abar_next))
            noise = torch.randn(shape, device=device, generator=generator) \
                if eta > 0 else torch.zeros_like(z)

            dir_zt = torch.sqrt((1 - abar_next - sigma ** 2).clamp(min=0.0)) * eps_hat
            z = abar_next.sqrt() * z0_hat + dir_zt + sigma * noise

        return z


## DiT 

In [ ]:
"""
Diffusion Transformer (DiT) for text-to-audio latent diffusion - pure PyTorch.

Follows DiT (Peebles & Xie, 2023) with two additions needed for this project:

  1. CROSS-ATTENTION to T5 text tokens inside every block (PixArt-alpha style),
     so captions steer generation at token granularity.
  2. A "REPA tap": the forward pass can return the hidden state after an early
     block, which train.py aligns with frozen AST features (REPA loss).

Block layout (repeated `depth` times):

    x -> [adaLN-Zero LN -> Self-Attention  -> gate] -> +residual
      -> [        LN    -> Cross-Attention -> gate] -> +residual
      -> [adaLN-Zero LN -> MLP             -> gate] -> +residual

adaLN-Zero: the timestep (+ pooled text) embedding regresses per-block
(shift, scale, gate) triples.  All gates are initialised to ZERO, so at step 0
every block is the identity function - a major training-stability win that we
keep from the original DiT.
"""


# --------------------------------------------------------------------------- #
#  Embedders                                                                   #
# --------------------------------------------------------------------------- #
class TimestepEmbedder(nn.Module):
    """Map a scalar diffusion timestep t to a `hidden_size` vector.

    Classic sinusoidal features (like Transformer positions) followed by a
    2-layer MLP.  The output is the global conditioning vector that drives
    every adaLN-Zero modulation in the network.
    """

    def __init__(self, hidden_size: int, freq_dim: int = 256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(
            nn.Linear(freq_dim, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    @staticmethod
    def sinusoidal(t: torch.Tensor, dim: int, max_period: int = 10_000) -> torch.Tensor:
        """t: [B] (float, any scale) -> [B, dim] sinusoidal features."""
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half
        )
        args = t.float()[:, None] * freqs[None]                    # [B, half]
        return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.sinusoidal(t, self.freq_dim))


def build_2d_sincos_pos_embed(dim: int, grid_t: int, grid_f: int) -> torch.Tensor:
    """Fixed 2-D sin/cos positional embedding over the (time, freq) token grid.

    The latent is spatial (time x frequency), so each axis gets half of the
    embedding dimensions.  Returns [grid_t * grid_f, dim], flattened
    time-major (token n = t * grid_f + f) to match PatchEmbed's flatten order.
    """
    assert dim % 4 == 0, "pos-embed dim must be divisible by 4 (2 axes x sin/cos)"

    def axis_embed(positions: torch.Tensor, axis_dim: int) -> torch.Tensor:
        omega = torch.arange(axis_dim // 2, dtype=torch.float32) / (axis_dim // 2)
        omega = 1.0 / (10_000 ** omega)                            # [axis_dim/2]
        out = positions.float()[:, None] * omega[None]             # [N, axis_dim/2]
        return torch.cat([torch.sin(out), torch.cos(out)], dim=-1)  # [N, axis_dim]

    t_pos = torch.arange(grid_t).repeat_interleave(grid_f)         # 0,0,..,1,1,..
    f_pos = torch.arange(grid_f).repeat(grid_t)                    # 0,1,..,0,1,..
    emb = torch.cat([axis_embed(t_pos, dim // 2), axis_embed(f_pos, dim // 2)], dim=-1)
    return emb                                                     # [T*F, dim]


class PatchEmbed(nn.Module):
    """Latent [B, C, T, F] -> token sequence [B, N, hidden] via p x p patches.

    Implemented as a strided Conv2d (the standard ViT trick): each p x p patch
    of the latent becomes exactly one token.
    """

    def __init__(self, in_channels: int, hidden_size: int, patch_size: int):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, hidden_size,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)                       # [B, hidden, T/p, F/p]
        return x.flatten(2).transpose(1, 2)    # [B, N, hidden]  (time-major)


# --------------------------------------------------------------------------- #
#  Attention primitives (hand-rolled, no external libs)                        #
# --------------------------------------------------------------------------- #
class SelfAttention(nn.Module):
    """Standard multi-head self-attention over the latent tokens."""

    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        # One fused projection, then split into Q, K, V and the head dimension.
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)   # each [B, heads, N, head_dim]
        x = F.scaled_dot_product_attention(q, k, v)   # flash/mem-efficient kernel
        x = x.transpose(1, 2).reshape(B, N, D)
        return self.out(x)


class CrossAttention(nn.Module):
    """Latent tokens attend to T5 text tokens.

    Queries come from the latent stream, keys/values from the (projected)
    text hidden states.  `text_mask` marks REAL tokens with 1 - padding
    positions are masked out of the attention so the model never conditions
    on pad garbage (a classic silent bug when forgotten).
    """

    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.out = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor, ctx: torch.Tensor,
                ctx_mask: torch.Tensor | None) -> torch.Tensor:
        B, N, D = x.shape
        L = ctx.shape[1]
        q = self.q(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        kv = self.kv(ctx).reshape(B, L, 2, self.num_heads, self.head_dim)
        k, v = kv.permute(2, 0, 3, 1, 4)       # each [B, heads, L, head_dim]

        attn_mask = None
        if ctx_mask is not None:
            # [B, L] {0,1} -> additive mask broadcast over heads and queries.
            attn_mask = torch.where(ctx_mask.bool(), 0.0, float("-inf"))
            attn_mask = attn_mask[:, None, None, :].to(q.dtype)

        x = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        x = x.transpose(1, 2).reshape(B, N, D)
        return self.out(x)


class MLP(nn.Module):
    def __init__(self, dim: int, hidden: int):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x), approximate="tanh"))


def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """adaLN modulation: LayerNorm output is shifted/scaled per sample."""
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


# --------------------------------------------------------------------------- #
#  DiT block                                                                   #
# --------------------------------------------------------------------------- #
class DiTBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float):
        super().__init__()
        # elementwise_affine=False: the affine part is provided by adaLN.
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.attn = SelfAttention(dim, num_heads)
        self.norm_ctx = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.cross = CrossAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = MLP(dim, int(dim * mlp_ratio))

        # adaLN-Zero head: regress (shift, scale, gate) for self-attn and MLP
        # from the conditioning vector -> 6 chunks of `dim`.
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.adaLN[-1].weight)   # ZERO init => block starts as
        nn.init.zeros_(self.adaLN[-1].bias)     # identity (adaLN-Zero).
        # Zero-init the cross-attention output too, so text conditioning fades
        # in smoothly instead of destabilising early training.
        nn.init.zeros_(self.cross.out.weight)
        nn.init.zeros_(self.cross.out.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor,
                ctx: torch.Tensor, ctx_mask: torch.Tensor | None) -> torch.Tensor:
        (shift_sa, scale_sa, gate_sa,
         shift_mlp, scale_mlp, gate_mlp) = self.adaLN(cond).chunk(6, dim=-1)

        # 1) self-attention (adaLN-Zero modulated + gated residual)
        x = x + gate_sa.unsqueeze(1) * self.attn(
            modulate(self.norm1(x), shift_sa, scale_sa))
        # 2) cross-attention to text (plain pre-LN residual, zero-init output)
        x = x + self.cross(self.norm_ctx(x), ctx, ctx_mask)
        # 3) MLP (adaLN-Zero modulated + gated residual)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(
            modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x


class FinalLayer(nn.Module):
    """adaLN-modulated LayerNorm -> linear projection back to patch pixels."""

    def __init__(self, dim: int, patch_size: int, out_channels: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(dim, patch_size * patch_size * out_channels)
        self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        # Zero init => the network initially predicts 0 everywhere, which is
        # the correct "do nothing" prior for a residual denoiser.
        nn.init.zeros_(self.adaLN[-1].weight)
        nn.init.zeros_(self.adaLN[-1].bias)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        shift, scale = self.adaLN(cond).chunk(2, dim=-1)
        return self.linear(modulate(self.norm(x), shift, scale))


# --------------------------------------------------------------------------- #
#  The full model                                                              #
# --------------------------------------------------------------------------- #
class DiT(nn.Module):
    """Text-conditioned Diffusion Transformer over VAE mel-latents.

    forward(z_t, t, text_emb, text_mask) -> prediction [B, C, T, F]
    (the prediction target - epsilon or v - is chosen by the diffusion code,
    the network is agnostic).
    """

    def __init__(self, latent_channels: int, latent_time: int, latent_freq: int,
                 patch_size: int, hidden_size: int, depth: int, num_heads: int,
                 mlp_ratio: float, text_dim: int, repa_layer: int):
        super().__init__()
        assert latent_time % patch_size == 0 and latent_freq % patch_size == 0
        self.out_channels = latent_channels
        self.patch_size = patch_size
        self.grid_t = latent_time // patch_size
        self.grid_f = latent_freq // patch_size
        self.num_tokens = self.grid_t * self.grid_f
        self.repa_layer = repa_layer

        self.patch_embed = PatchEmbed(latent_channels, hidden_size, patch_size)
        # Fixed (non-trained) 2-D sincos positional embedding.
        self.register_buffer(
            "pos_embed",
            build_2d_sincos_pos_embed(hidden_size, self.grid_t, self.grid_f)[None],
            persistent=False,
        )

        self.t_embed = TimestepEmbedder(hidden_size)
        # Trainable projection: T5 hidden size -> DiT width (the "glue" layer).
        self.text_proj = nn.Sequential(
            nn.LayerNorm(text_dim),
            nn.Linear(text_dim, hidden_size),
        )
        # Pooled-text -> conditioning vector: gives every adaLN a global text
        # signal in addition to the per-token cross-attention.
        self.pooled_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        # Learned NULL text sequence for classifier-free guidance (one token).
        # Training replaces the caption with this with prob p_uncond; sampling
        # uses it as the unconditional branch of CFG.
        self.null_text = nn.Parameter(torch.zeros(1, 1, hidden_size))

        self.blocks = nn.ModuleList(
            DiTBlock(hidden_size, num_heads, mlp_ratio) for _ in range(depth)
        )
        self.final = FinalLayer(hidden_size, patch_size, self.out_channels)
        self._init_weights()

    def _init_weights(self):
        # Xavier for generic linears; the zero-inits inside DiTBlock/FinalLayer
        # were applied at construction and are re-applied after this pass.
        def basic(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        self.apply(basic)
        # Patchify conv: treat as a linear layer over flattened patches.
        w = self.patch_embed.proj.weight
        nn.init.xavier_uniform_(w.view(w.shape[0], -1))
        nn.init.zeros_(self.patch_embed.proj.bias)
        nn.init.normal_(self.t_embed.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embed.mlp[2].weight, std=0.02)
        for block in self.blocks:           # restore the adaLN-Zero contract
            nn.init.zeros_(block.adaLN[-1].weight)
            nn.init.zeros_(block.adaLN[-1].bias)
            nn.init.zeros_(block.cross.out.weight)
            nn.init.zeros_(block.cross.out.bias)
        nn.init.zeros_(self.final.adaLN[-1].weight)
        nn.init.zeros_(self.final.adaLN[-1].bias)
        nn.init.zeros_(self.final.linear.weight)
        nn.init.zeros_(self.final.linear.bias)

    # ------------------------------------------------------------------ #
    def null_context(self, batch_size: int) -> tuple[torch.Tensor, torch.Tensor]:
        """The learned unconditional context (already in DiT width) + mask."""
        ctx = self.null_text.expand(batch_size, -1, -1)
        mask = torch.ones(batch_size, 1, device=ctx.device, dtype=torch.long)
        return ctx, mask

    def unpatchify(self, x: torch.Tensor) -> torch.Tensor:
        """[B, N, p*p*C] tokens -> [B, C, T, F] latent grid."""
        B = x.shape[0]
        p, c = self.patch_size, self.out_channels
        x = x.reshape(B, self.grid_t, self.grid_f, p, p, c)
        x = torch.einsum("btfpqc->bctpfq", x)           # interleave patch pixels
        return x.reshape(B, c, self.grid_t * p, self.grid_f * p)

    def forward(self, z_t: torch.Tensor, t: torch.Tensor,
                text_emb: torch.Tensor | None, text_mask: torch.Tensor | None,
                drop_mask: torch.Tensor | None = None,
                return_repa_hidden: bool = False):
        """
        z_t        : [B, C, T, F]  noisy latent
        t          : [B]           diffusion timestep (float in [0, T))
        text_emb   : [B, L, text_dim] T5 hidden states (None => unconditional)
        text_mask  : [B, L]        1 = real token, 0 = padding
        drop_mask  : [B] bool      True => replace this sample's caption with
                                   the null context (CFG dropout, train only)
        """
        B = z_t.shape[0]
        x = self.patch_embed(z_t) + self.pos_embed      # [B, N, hidden]

        # --- build the text context in DiT width -------------------------- #
        if text_emb is None:
            ctx, ctx_mask = self.null_context(B)
        else:
            ctx = self.text_proj(text_emb)              # [B, L, hidden]
            ctx_mask = text_mask
            # NOTE: no `.any()` short-circuit here - reading a device tensor
            # from Python forces a host<->device sync every step, which stalls
            # XLA/TPU pipelines.  torch.where with an all-False mask is free.
            if drop_mask is not None:
                # Per-sample CFG dropout: pad the null token out to length L so
                # conditional/unconditional samples share one tensor shape.
                null = self.null_text.expand(B, ctx.shape[1], -1)
                ctx = torch.where(drop_mask[:, None, None], null, ctx)
                null_mask = torch.zeros_like(ctx_mask)
                null_mask[:, 0] = 1                     # only token 0 is "real"
                ctx_mask = torch.where(drop_mask[:, None], null_mask, ctx_mask)

        # --- global conditioning vector (timestep + pooled text) ---------- #
        cond = self.t_embed(t)
        if ctx_mask is not None:
            denom = ctx_mask.sum(dim=1, keepdim=True).clamp(min=1)
            pooled = (ctx * ctx_mask.unsqueeze(-1)).sum(dim=1) / denom
        else:
            pooled = ctx.mean(dim=1)
        cond = cond + self.pooled_proj(pooled)

        # --- transformer trunk with optional REPA tap ---------------------- #
        repa_hidden = None
        for i, block in enumerate(self.blocks):
            x = block(x, cond, ctx, ctx_mask)
            if return_repa_hidden and i == self.repa_layer:
                repa_hidden = x                          # [B, N, hidden]

        out = self.unpatchify(self.final(x, cond))       # [B, C, T, F]
        if return_repa_hidden:
            return out, repa_hidden
        return out


# --------------------------------------------------------------------------- #
#  REPA projection head (trainable, train-time only, dropped at inference)     #
# --------------------------------------------------------------------------- #
class RepaProjector(nn.Module):
    """3-layer MLP mapping DiT hidden tokens -> AST feature space.

    REPA aligns g(h_l) with the frozen encoder's features of the CLEAN input
    via a per-token cosine loss.  This head (and the loss) exist only during
    training; sampling never touches them.
    """

    def __init__(self, dit_dim: int, target_dim: int, hidden: int = 2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dit_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, target_dim),
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h)


def repa_loss(projected: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Negative mean per-token cosine similarity.

    projected : [B, N, D] = g(h_l)   (trainable path)
    target    : [B, N, D] = y*       (frozen AST features, precomputed on the
                                      same N-token grid as the DiT)
    """
    projected = F.normalize(projected, dim=-1)
    target = F.normalize(target.float(), dim=-1)
    return -(projected * target).sum(dim=-1).mean()


## EMA (Exponential Moving Average of model weights)

In [ ]:
class EMA:
    """Keeps a slow-moving copy of the DiT.  Sampling from the EMA weights
    instead of the raw weights is a standard, material quality win."""

    def __init__(self, model: torch.nn.Module, decay: float):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: torch.nn.Module):
        for ema_p, p in zip(self.model.parameters(), model.parameters()):
            ema_p.lerp_(p, 1.0 - self.decay)     # ema = d*ema + (1-d)*param
        for ema_b, b in zip(self.model.buffers(), model.buffers()):
            ema_b.copy_(b)

## Build the Model

In [ ]:
def build_model(cfg: Config) -> DiT:
    return DiT(
        latent_channels=cfg.latent.channels,
        latent_time=cfg.latent.time,
        latent_freq=cfg.latent.freq,
        patch_size=cfg.dit.patch_size,
        hidden_size=cfg.dit.hidden_size,
        depth=cfg.dit.depth,
        num_heads=cfg.dit.num_heads,
        mlp_ratio=cfg.dit.mlp_ratio,
        text_dim=cfg.pretrained.text_dim,
        repa_layer=cfg.dit.repa_layer,
    )


## Training

In [ ]:
def lr_lambda_factory(warmup: int, total: int):
    def fn(step: int) -> float:
        if step < warmup:
            return step / max(warmup, 1)
        progress = (step - warmup) / max(total - warmup, 1)
        return 0.5 * (1.0 + torch.cos(torch.tensor(progress * torch.pi)).item())
    return fn


def repa_lambda(step: int, base: float, decay_steps: int) -> float:
    """REPA weight schedule: full strength early, linear decay to 0."""
    if base <= 0.0:
        return 0.0
    return base * max(0.0, 1.0 - step / max(decay_steps, 1))


def collate_tensors_only(batch: list[dict]) -> dict:
    """Like dataset.collate but WITHOUT caption strings (TPU-safe)."""
    return {
        "latent": torch.stack([b["latent"] for b in batch]),
        "text_emb": torch.stack([b["text_emb"] for b in batch]),
        "text_mask": torch.stack([b["text_mask"] for b in batch]),
        "repa": torch.stack([b["repa"] for b in batch]),
    }


def save_ckpt(
    out_dir: Path,
    model,
    projector,
    ema,
    opt,
    sched,
    step: int,
    cfg,
    *,
    hf_repo_id: str | None = None,
    hf_token: str | None = None,
    upload_every: int = 5,
    keep_local: int = 1,
    path_in_repo: str = "checkpoints",
    force_upload: bool = False,
):
    """
    Save checkpoint, prune old local files, optionally upload to Hugging Face.

    - Writes: out_dir / f"ckpt_{step:07d}.pt"
    - Deletes older ckpt_*.pt so /kaggle/working stays small
    - Uploads every `upload_every` local saves (or always if force_upload=True)
    """
    out_dir = Path(out_dir)
    ckpt_path = out_dir / f"ckpt_{step:07d}.pt"
    is_master = xm.is_master_ordinal()

    payload = {
        "model": model.state_dict(),
        "projector": projector.state_dict(),
        "ema": ema.model.state_dict(),
        "opt": opt.state_dict(),
        "sched": sched.state_dict(),
        "step": step,
        "config": cfg.as_dict(),
    }
    xm.save(payload, str(ckpt_path))

    if is_master:
        out_dir.mkdir(parents=True, exist_ok=True)

        ckpts = sorted(out_dir.glob("ckpt_*.pt"))
        if keep_local >= 0 and len(ckpts) > keep_local:
            for old in ckpts[:-keep_local]:
                try:
                    old.unlink()
                    print(f"[ckpt] deleted old local: {old.name}", flush=True)
                except OSError as e:
                    print(f"[ckpt] failed to delete {old}: {e}", flush=True)

        token_ok = bool(hf_token)  # treat "" as missing
        should_upload = (
            bool(hf_repo_id)
            and token_ok
            and upload_every > 0
            and (force_upload or (step // max(cfg.train.ckpt_every, 1)) % upload_every == 0)
        )
        if should_upload:
            stage = out_dir / "_hf_upload"
            stage.mkdir(exist_ok=True)
            for f in stage.glob("*"):
                if f.is_file():
                    f.unlink()
            staged = stage / ckpt_path.name
            shutil.copy2(ckpt_path, staged)
            upload_ckpt_folder(
                local_dir=stage,
                repo_id=hf_repo_id,
                token=hf_token,
                path_in_repo=path_in_repo,
                commit_message=f"ckpt step {step}",
            )

    xm.rendezvous("ckpt_saved")


# --------------------------------------------------------------------------- #
#  Per-process entry point (runs 8x on a v5e-8)
# --------------------------------------------------------------------------- #
def _mp_fn(index: int, data, out, resume, workers,
           hf_repo, hf_token, upload_every, keep_local, path_in_repo,
           wandb_project, wandb_run_name, wandb_enabled):
    cfg = Config()
    device = xm.xla_device()
    rank = xr.global_ordinal()
    world = xr.world_size()
    is_master = xm.is_master_ordinal()

    if is_master:
        print(f"[xla] world_size={world}  per-chip batch={cfg.train.batch_size}  "
              f"global batch={cfg.train.batch_size * world}")

    # ---------------- data (sharded across the 8 chips) -------------------- #
    ds = PrecomputedAudioCaps(data, "train")
    sampler = DistributedSampler(ds, num_replicas=world, rank=rank,
                                 shuffle=True, drop_last=True)
    loader = DataLoader(ds, batch_size=cfg.train.batch_size, sampler=sampler,
                        num_workers=workers, drop_last=True,
                        collate_fn=collate_tensors_only,
                        persistent_workers=workers > 0)
    device_loader = pl.MpDeviceLoader(loader, device)
    if is_master:
        print(f"[xla] {len(ds)} samples, {len(loader)} steps/epoch/chip, "
              f"latent scale = {ds.latent_scale:.4f}")

    # ---------------- model / optimizer ------------------------------------ #
    # Same seed on every rank BEFORE building the model.
    torch.manual_seed(cfg.train.seed)
    model = build_model(cfg).to(device)
    projector = RepaProjector(cfg.dit.hidden_size,
                              cfg.pretrained.repa_dim).to(device)
    diffusion = Diffusion(cfg.diffusion.num_train_steps, cfg.diffusion.schedule,
                          cfg.diffusion.logit_normal_mean,
                          cfg.diffusion.logit_normal_std).to(device)

    if is_master:
        n_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"[xla] DiT parameters: {n_params:.1f}M")

    params = list(model.parameters()) + list(projector.parameters())
    opt = torch.optim.AdamW(params, lr=cfg.train.lr, betas=(0.9, 0.95),
                            weight_decay=cfg.train.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda_factory(cfg.train.warmup_steps, cfg.train.total_steps))
    ema = EMA(model, cfg.train.ema_decay)

    start_step = 0
    if resume:
        ckpt = torch.load(resume, map_location="cpu")
        model.load_state_dict(ckpt["model"])
        projector.load_state_dict(ckpt["projector"])
        ema.model.load_state_dict(ckpt["ema"])
        opt.load_state_dict(ckpt["opt"])
        sched.load_state_dict(ckpt["sched"])
        start_step = ckpt["step"] + 1
        if is_master:
            print(f"[xla] resumed from {resume} at step {start_step}")

    # Different RNG per rank for noise / timesteps / CFG dropout.
    torch.manual_seed(cfg.train.seed * 1000 + rank)
    xm.set_rng_state(cfg.train.seed * 1000 + rank, device=device)

    out_dir = Path(out)
    if is_master:
        out_dir.mkdir(parents=True, exist_ok=True)

    # ---------------- W&B + logging ---------------------------------------- #
    init_wandb(cfg, is_master, project=wandb_project,
               run_name=wandb_run_name, enabled=wandb_enabled)
    t0 = time.time()
    log_fn = make_log_fn(is_master, start_step, t0, sched,
                         use_wandb=wandb_enabled)

    # ---------------- the loop --------------------------------------------- #
    model.train()
    step = start_step
    epoch = 0
    done = False

    while not done:
        sampler.set_epoch(epoch)
        for batch in device_loader:
            if step >= cfg.train.total_steps:
                done = True
                break

            z0 = batch["latent"]
            text_emb = batch["text_emb"]
            text_mask = batch["text_mask"]
            y_star = batch["repa"]
            B = z0.shape[0]

            drop_mask = torch.rand(B, device=device) < cfg.diffusion.p_uncond
            t = diffusion.sample_timesteps(B, device)
            eps = torch.randn_like(z0)
            z_t = diffusion.add_noise(z0, t, eps)
            v_target = diffusion.v_target(z0, t, eps)

            lam = repa_lambda(step, cfg.train.repa_weight,
                              cfg.train.repa_decay_steps)

            with torch.autocast("xla", dtype=torch.bfloat16):
                v_pred, h_l = model(z_t, t.float(), text_emb, text_mask,
                                    drop_mask=drop_mask,
                                    return_repa_hidden=True)
                loss_diff = F.mse_loss(v_pred.float(), v_target)
                loss_repa = repa_loss(projector(h_l).float(), y_star)
                loss = loss_diff + lam * loss_repa

            opt.zero_grad(set_to_none=True)
            loss.backward()
            xm.reduce_gradients(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg.train.grad_clip)
            opt.step()
            sched.step()
            ema.update(model)

            if step % cfg.train.log_every == 0:
                xm.add_step_closure(
                    log_fn, args=(step, loss.detach(), loss_diff.detach(),
                                  loss_repa.detach(), lam))

            if step > 0 and step % cfg.train.ckpt_every == 0:
                save_ckpt(
                    out_dir,
                    model, projector, ema, opt, sched, step, cfg,
                    hf_repo_id=hf_repo,
                    hf_token=hf_token,
                    upload_every=upload_every,
                    keep_local=keep_local,
                    path_in_repo=path_in_repo,
                )
                if is_master:
                    print(f"[xla] checkpoint at step {step}", flush=True)

            step += 1
        epoch += 1

    # Final save + force HF upload so the last weights are durable.
    save_ckpt(
        out_dir,
        model, projector, ema, opt, sched, step - 1, cfg,
        hf_repo_id=hf_repo,
        hf_token=hf_token,
        upload_every=upload_every,
        keep_local=keep_local,
        path_in_repo=path_in_repo,
        force_upload=True,
    )
    if is_master:
        if wandb.run is not None:
            wandb.finish()
        print(f"[xla] done - final checkpoint in {out_dir}", flush=True)


In [ ]:
# Launch data-parallel training on all 8 TPU chips.
# HF / W&B settings come from the config cell above.
try:
    torch_xla.launch(
        _mp_fn,
        args=(
            DATA_DIR, OUT_DIR, RESUME, WORKERS,
            HF_REPO, HF_TOKEN, UPLOAD_EVERY, KEEP_LOCAL, HF_PATH_IN_REPO,
            WANDB_PROJECT, WANDB_RUN_NAME, WANDB_ENABLED,
        ),
    )
except AttributeError:
    import torch_xla.distributed.xla_multiprocessing as xmp
    xmp.spawn(
        _mp_fn,
        args=(
            DATA_DIR, OUT_DIR, RESUME, WORKERS,
            HF_REPO, HF_TOKEN, UPLOAD_EVERY, KEEP_LOCAL, HF_PATH_IN_REPO,
            WANDB_PROJECT, WANDB_RUN_NAME, WANDB_ENABLED,
        ),
    )
